In [ ]:
# -------------------------------------------------------------
#  Import Libraries
# -------------------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

sns.set_theme(style="whitegrid")

In [ ]:
# -------------------------------------------------------------
# Load and Clean Data (Target & Outliers)
# -------------------------------------------------------------
df = pd.read_csv("AB_NYC_2019.csv")

# Ensure price is clean float and positive
if df["price"].dtype == object:
    df["price"] = df["price"].astype(str).str.replace(r'[\$,]', '', regex=True).astype(float)
df = df[df["price"] > 0].copy()

# Remove extreme price outliers via IQR
Q1 = df["price"].quantile(0.25)
Q3 = df["price"].quantile(0.75)
IQR = Q3 - Q1
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

df = df[(df["price"] >= lower_limit) & (df["price"] <= upper_limit)].copy()
print(f"Dataset shape after cleaning: {df.shape}")

Dataset shape after cleaning: (45912, 16)


In [ ]:
# -------------------------------------------------------------
# Feature Selection & Train-Test Split
# -------------------------------------------------------------
num_cols = [
    "latitude", 
    "longitude", 
    "minimum_nights", 
    "number_of_reviews", 
    "reviews_per_month", 
    "calculated_host_listings_count", 
    "availability_365"
]

cat_cols = [
    "neighbourhood_group", 
    "room_type"
]

X = df[num_cols + cat_cols].copy()
y = df["price"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

X_train shape: (36729, 9), X_test shape: (9183, 9)


In [ ]:
# -------------------------------------------------------------
# Preprocessing
# -------------------------------------------------------------
# Fill missing numerical values in training data
imputer_values = X_train[num_cols].median()
X_train_num_filled = X_train[num_cols].fillna(imputer_values)
X_test_num_filled = X_test[num_cols].fillna(imputer_values)

# Scaler
num_scaler = StandardScaler()
X_train_num_scaled = num_scaler.fit_transform(X_train_num_filled)
X_test_num_scaled = num_scaler.transform(X_test_num_filled)

# Categorical: Fill missing & OneHotEncode
X_train_cat_filled = X_train[cat_cols].fillna("Missing")
X_test_cat_filled = X_test[cat_cols].fillna("Missing")

cat_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_train_cat_encoded = cat_encoder.fit_transform(X_train_cat_filled)
X_test_cat_encoded = cat_encoder.transform(X_test_cat_filled)

# Combined arrays for training
X_train_final = np.hstack([X_train_num_scaled, X_train_cat_encoded])
X_test_final = np.hstack([X_test_num_scaled, X_test_cat_encoded])



In [18]:

models = {
    "Linear Regression (Baseline)": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting Regressor": GradientBoostingRegressor(n_estimators=100, random_state=42)
}

comparison_records = []

for name, model in models.items():
    # Fit model on the scaled & encoded training set
    model.fit(X_train_final, y_train)
    
    # Generate predictions
    train_preds = model.predict(X_train_final)
    test_preds = model.predict(X_test_final)
    
    # Calculate evaluation metrics
    mae = mean_absolute_error(y_test, test_preds)
    rmse = np.sqrt(mean_squared_error(y_test, test_preds))
    r2 = r2_score(y_test, test_preds)
    train_r2 = r2_score(y_train, train_preds)
    
    comparison_records.append({
        "Model": name,
        "Train R²": round(train_r2, 4),
        "Test R²": round(r2, 4),
        "Test MAE ($)": round(mae, 2),
        "Test RMSE ($)": round(rmse, 2)
    })

# Convert to DataFrame for clear comparison
comparison_df = pd.DataFrame(comparison_records)
display(comparison_df)

,Model,Train R²,Test R²,Test MAE ($),Test RMSE ($)
0,Linear Regression (Baseline),0.4837,0.4791,36.20,48.57
1,Random Forest Regressor,0.9397,0.5740,31.67,43.92
2,Gradient Boosting Regressor,0.5727,0.5613,32.51,44.57


In [ ]:
# -------------------------------------------------------------
#  Model Training & Hyperparameter Tuning
# -------------------------------------------------------------
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

param_dist = {
    "n_estimators": [100, 150, 200],
    "max_depth": [10, 15, 20, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

rf_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=6,
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1
)

rf_search.fit(X_train_final, y_train)
best_model = rf_search.best_estimator_

# Evaluate on test data
test_preds = best_model.predict(X_test_final)
print(f"MAE:  ${mean_absolute_error(y_test, test_preds):.2f}")
print(f"RMSE: ${np.sqrt(mean_squared_error(y_test, test_preds)):.2f}")
print(f"R²:   {r2_score(y_test, test_preds):.4f}")

MAE:  $31.46
RMSE: $43.71
R²:   0.5781


In [ ]:

export_data = {
    # Trained regressor
    "model": best_model,
    
    # Raw scaler values (plain numpy arrays, zero version dependency)
    "scaler_mean": num_scaler.mean_,
    "scaler_scale": num_scaler.scale_,
    
    # Feature & Category lists
    "num_cols": num_cols,
    "cat_cols": cat_cols,
    "categories": [list(c) for c in cat_encoder.categories_]
}

joblib.dump(export_data, "airbnb_price_artifacts.joblib")
print("Saved cleanly to airbnb_price_artifacts.joblib!")

Saved cleanly to airbnb_price_artifacts.joblib!
